In [1]:
import torch
from torch import nn 
from torch import optim
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy

### Linear Regressor Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

# 1. Dataset Generation
torch.manual_seed(204)

batch_size = 1000
feature_dim = 10 

X = torch.randn((batch_size, feature_dim))
weights = torch.randn(feature_dim)
bias = torch.randn(1)

# Added noise for a realistic regression task
noise = torch.randn(batch_size, 1) * 0.1 
Y = torch.matmul(X, weights.unsqueeze(1)) + bias + noise

# Split data (750 Train, 150 Val, 100 Test)
X_train, X_val, X_test = torch.split(X, [750, 150, 100])
Y_train, Y_val, Y_test = torch.split(Y, [750, 150, 100])

# 2. Model Definition
class LinearRegressor(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.linear_layer = nn.Linear(feature_dim, 1)

    def forward(self, X):
        return self.linear_layer(X)

regressor = LinearRegressor(feature_dim)

# 3. Training Setup
epochs = 1000
learning_rate = 0.01
optimizer = optim.Adam(regressor.parameters(), lr=learning_rate)
loss_func = nn.MSELoss()

best_val_loss = float("inf")
patience = 5
counter = 0 
best_state = None

# 4. Training Loop
for i in range(epochs):
    # --- Training Phase ---
    regressor.train()
    optimizer.zero_grad()
    
    train_predicted = regressor(X_train)
    train_loss = loss_func(train_predicted, Y_train)
    
    train_loss.backward()
    optimizer.step()

    # --- Validation Phase ---
    regressor.eval()
    with torch.no_grad():
        val_predicted = regressor(X_val)
        val_loss = loss_func(val_predicted, Y_val)
    
    val = val_loss.item()
    
    # --- Early Stopping Logic ---
    if val < best_val_loss:
        best_val_loss = val
        counter = 0 
        best_state = deepcopy(regressor.state_dict())
    else:
        counter += 1

    if i % 10 == 0:
        print(f"Epoch {i:3d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val:.4f}")
        
    if counter >= patience:
        print(f"Early stopping at epoch {i}")
        break

# 5. Testing Phase
if best_state is not None:
    regressor.load_state_dict(best_state)
    print("Restored best model weights.")

regressor.eval()
with torch.no_grad():
    test_predicted = regressor(X_test)
    test_loss = loss_func(test_predicted, Y_test)

    print(f"Final Test Loss: {test_loss.item():.4f}")


## Logistic Regression Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

# 1. Data Generation
torch.manual_seed(204)
batch_size = 1024 
input_dim = 8

X = torch.randn((batch_size, input_dim))
weights = torch.randn(input_dim)
bias = torch.randn(1)
noise = torch.randn((batch_size, 1))

# FIX: Add noise to the logits before thresholding to make it a realistic ML problem
logits = torch.matmul(X, weights.unsqueeze(1)) + bias + noise
Y = (torch.sigmoid(logits) >= 0.5).float().view(-1, 1)

X_train, X_val, X_test = torch.split(X, [750, 150, 124]) 
Y_train, Y_val, Y_test = torch.split(Y, [750, 150, 124])

# 2. Model
class LogisticRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear_layer = nn.Linear(input_dim, 1)

    def forward(self, X):
        # Returns Logits (unnormalized scores)
        return self.linear_layer(X)

logistic_regressor = LogisticRegressor(input_dim) 

# 3. Setup
optimizer = optim.Adam(logistic_regressor.parameters(), lr=0.01)
loss_func = nn.BCEWithLogitsLoss() # Handles Sigmoid internally

epochs = 1000 
patience = 20 
counter = 0 
min_loss = float("inf")
best_state = None 

# Helper function to calculate accuracy
def calculate_accuracy(y_pred_logits, y_true):
    # Apply sigmoid to convert logits to probabilities
    probs = torch.sigmoid(y_pred_logits)
    # Round to get 0 or 1
    predictions = probs.round()
    correct = (predictions == y_true).float()
    return correct.mean()

# 4. Training Loop
for i in range(epochs):
    # --- Train ---
    logistic_regressor.train()
    optimizer.zero_grad()

    predicted_train = logistic_regressor(X_train)
    train_loss = loss_func(predicted_train, Y_train)

    train_loss.backward()
    optimizer.step()

    # --- Validation ---
    logistic_regressor.eval()
    with torch.no_grad():
        predicted_val = logistic_regressor(X_val)
        val_loss = loss_func(predicted_val, Y_val)
        val_acc = calculate_accuracy(predicted_val, Y_val)

        val = val_loss.item()
        
        # Early Stopping Check
        if val < min_loss:
            min_loss = val 
            best_state = deepcopy(logistic_regressor.state_dict())
            counter = 0 
        else:
            counter += 1

        if i % 10 == 0:
            print(f"Epoch {i:3d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val:.4f} | Val Acc: {val_acc:.4f}")
            
        if counter >= patience:
            print(f"Early stopping at epoch {i}")
            break

# 5. Restore Best Weights and Test
if best_state is not None:
    logistic_regressor.load_state_dict(best_state)
    print("Restored best model weights.")

logistic_regressor.eval()
with torch.no_grad():
    predicted_test = logistic_regressor(X_test)
    test_loss = loss_func(predicted_test, Y_test)
    test_acc = calculate_accuracy(predicted_test, Y_test)

    print(f"Final Test Loss: {test_loss.item():.4f}")
    print(f"Final Test Accuracy: {test_acc:.4f}")


## Implement MLP with multi class classification

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

# 1. Setup & Data Generation
feature_dim = 10 
batch_size = 2048 
hidden_dim = 256
num_classes = 3 

torch.manual_seed(204)

# Generate synthetic data
X = torch.randn((batch_size, feature_dim))

# Simulate a non-linear relationship for the labels
weight1 = torch.randn((feature_dim, hidden_dim))
weight2 = torch.randn((hidden_dim, num_classes))
bias1 = torch.randn(1)
bias2 = torch.randn(1) 

# Create logits and then convert to class labels (0, 1, 2)
Y_logits = torch.relu(X @ weight1 + bias1) @ weight2 + bias2 
labels = torch.argmax(Y_logits, dim=1)

# Split Data
# Convention: Train (for optimization), Val (for early stopping), Test (final check)
# Renamed from your code to match standard ML conventions
X_train, X_val, X_test = torch.split(X, [1640, 205, 203])
Y_train, Y_val, Y_test = torch.split(labels, [1640, 205, 203])

# 2. Model Definition
class MultiClassifier(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim, bias=True),
            nn.ReLU(),
            # Output layer must have 'num_classes' neurons
            # No Softmax here because CrossEntropyLoss handles it
            nn.Linear(hidden_dim, num_classes, bias=True)     
        )
        
    def forward(self, X):
        return self.layers(X)

multiclassifier = MultiClassifier(feature_dim, hidden_dim, num_classes)

# 3. Training Setup
optimizer = optim.Adam(multiclassifier.parameters(), lr=0.0005)
loss_func = nn.CrossEntropyLoss() # Expects Logits + Class Indices
epochs = 2000 
patience = 30 
counter = 0 
best_state = None 
best_loss = float("inf")

# Helper for accuracy
def get_accuracy(logits, targets):
    predictions = torch.argmax(logits, dim=1)
    correct = (predictions == targets).sum().item()
    return correct / targets.size(0)

# 4. Training Loop
for i in range(epochs):
    # --- Train ---
    multiclassifier.train() # Fixed typo here
    optimizer.zero_grad()
    
    predicted_train = multiclassifier(X_train)
    loss_train = loss_func(predicted_train, Y_train)

    loss_train.backward()
    optimizer.step()

    # --- Validation (Early Stopping) ---
    multiclassifier.eval() # Fixed typo here
    with torch.no_grad():
        predicted_val = multiclassifier(X_val) # Using X_val here
        loss_val = loss_func(predicted_val, Y_val)
        val = loss_val.item()
        
        if val < best_loss:
            best_loss = val
            best_state = deepcopy(multiclassifier.state_dict())
            counter = 0
        else:
            counter += 1

    if i % 10 == 0:
        train_acc = get_accuracy(predicted_train, Y_train)
        val_acc = get_accuracy(predicted_val, Y_val)
        print(f"Epoch {i:4d} | Train Loss: {loss_train.item():.4f} (Acc: {train_acc:.2f}) | Val Loss: {val:.4f} (Acc: {val_acc:.2f})")

    if counter == patience:
        print(f"Early stopping at epoch {i}")
        break

# 5. Final Test
if best_state is not None:
    multiclassifier.load_state_dict(best_state)
    print("Restored best model weights.")

multiclassifier.eval() # Fixed typo here
with torch.no_grad():
    # Using X_test here for final evaluation
    prob_test = multiclassifier(X_test)
    loss_test = loss_func(prob_test, Y_test)
    accuracy = get_accuracy(prob_test, Y_test)
    
    print(f"Final Test Loss: {loss_test.item():.4f}; Accuracy: {accuracy:.4f}")


## embedding model with linear classifer  (NLP-style)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy

# --- 1. Robust Device Selection ---
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Check for Apple Silicon (M1/M2/M3)
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Running on: {device}")

# --- 2. Model Definitions (Unchanged) ---
class MaxPoolNLP(nn.Module):
    def __init__(self, vocab_size, embed_size, num_classes):
        super().__init__()
        self.embed_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=0)
        self.linear = nn.Linear(embed_size, num_classes, bias=True)

    def forward(self, input_ids, mask):
        embedding = self.embed_layer(input_ids)
        mask = mask.unsqueeze(-1)
        masked_embedding = embedding.masked_fill(mask==0, -float("inf"))
        max_pooled, _ = masked_embedding.max(dim=1) 

        # 3. Safety Handling (Optional but recommended):
        # If a sequence was entirely padding, max_pooled is now -inf.
        # Passing -inf to Linear layers causes NaNs. We clamp the RESULT, not the mask.
        # We replace -inf with 0.0 or a large negative number just to keep the math stable.
        max_pooled = torch.nan_to_num(max_pooled, nan=0.0, neginf=0.0)

        return self.linear(max_pooled) 

class MeanPoolNLP(nn.Module):
    def __init__(self, vocab_size, embed_size, num_classes):
        super().__init__()
        self.embed_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=0)
        self.linear = nn.Linear(embed_size, num_classes, bias=True)

    def forward(self, input_ids, mask):
        embedding = self.embed_layer(input_ids)
        mask = mask.unsqueeze(-1).float()
        masked_embedding = embedding * mask
        total_count = mask.sum(dim=1).clamp(min=1e-8)
        total_sum = masked_embedding.sum(dim=1) 
        mean_pooled = total_sum / total_count 
        return self.linear(mean_pooled)

def calculate_accuracy(predicted, label):
    return (predicted.argmax(dim=1) == label).float().mean().item()

# --- 3. Updated Training Function with Device Support ---
def train_test(model, loss_func, optimizer, train_loader, val_loader, epoch):
    # Move the entire model to the device (GPU/MPS/CPU)
    model = model.to(device)
    
    lowest_loss = float("inf")
    best_state = None
    patience = 20 
    count = 0
    
    for i in range(epoch):
        model.train()

        train_loss_accum = 0 
        train_acc_accum = 0
        train_batches = 0 
        
        # Iterate over batches
        for x_batch, mask_batch, y_batch in train_loader:
            # Move batch data to device
            x_batch = x_batch.to(device)
            mask_batch = mask_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            train_predicted = model(x_batch, mask_batch)
            train_loss = loss_func(train_predicted, y_batch)
            train_loss.backward()
            optimizer.step()

            train_loss_accum += train_loss.item()
            train_acc_accum += calculate_accuracy(train_predicted, y_batch)
            train_batches += 1

        avg_train_loss = train_loss_accum /train_batches
        avg_train_acc = train_acc_accum/train_batches 

        # Validation
        model.eval()
        val_loss_accum = 0
        val_acc_accum = 0 
        val_batches = 0
        
        with torch.no_grad():
            for x_val, mask_val, y_val in val_loader:
                x_val = x_val.to(device)
                mask_val = mask_val.to(device)
                y_val = y_val.to(device)
                
                val_predicted = model(x_val, mask_val)
                val_loss = loss_func(val_predicted, y_val)
                val_loss_accum += val_loss.item()
                val_acc_accum += calculate_accuracy(val_predicted, y_val)
                val_batches += 1
            
            avg_val_loss = val_loss_accum / val_batches
            avg_val_accuracy = val_acc_accum/val_batches

            if avg_val_loss < lowest_loss:
                lowest_loss = avg_val_loss 
                best_state = deepcopy(model.state_dict())
                count = 0
            else:
                count += 1

        if i % 10 == 0:
            print(f"Epoch {i}: train_loss: {avg_train_loss:.4f} (acc: {avg_train_acc:.2f}); val_loss: {avg_val_loss:.4f} (acc: {avg_val_accuracy:.2f})")

        if count == patience:
            print(f"Early stopping with epoch {i}")
            break
            
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def evaluate_model(model, loss_func, data_loader):
    model.eval() # Fixed syntax error from original code
    model = model.to(device)
    
    total_acc = 0
    total_loss = 0 
    batches = 0
    
    with torch.no_grad():
        for x, mask, y in data_loader:
            x, mask, y = x.to(device), mask.to(device), y.to(device)
            predicted = model(x, mask)
            total_loss += loss_func(predicted, y).item()
            total_acc += calculate_accuracy(predicted, y)
            batches += 1
        avg_loss = total_loss / batches 
    print(f"Final Accuracy: {total_acc/batches:.4f}; Final Loss: {avg_loss:.4f}")

# --- 4. Data Setup ---
torch.manual_seed(204)
batch_size = 1024 # You can lower this if you hit memory errors on GPU
seq_len = 10 
embed_size = 512 
vocab_size = 10000
num_classes = 3 

X = torch.randint(1, vocab_size, (batch_size, seq_len))
mask = torch.ones_like(X)

for i in range(batch_size):
    pad_len = torch.randint(0, seq_len//2, (1,)).item()
    if pad_len > 0:
        X[i,seq_len - pad_len:] = 0 
        mask[i, seq_len - pad_len:] = 0 

Y = torch.randint(0, num_classes, (batch_size,))

X_train, X_val, X_test = torch.split(X, [800,124,100])
Y_train, Y_val, Y_test = torch.split(Y, [800,124,100]) 
mask_train, mask_val, mask_test = torch.split(mask, [800,124,100])

# Create DataLoaders
train_data = TensorDataset(X_train, mask_train, Y_train)
val_data = TensorDataset(X_val, mask_val, Y_val)
test_data = TensorDataset(X_test, mask_test, Y_test)

# Note: shuffle=True is important for training!
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

max_pool_nlp = MaxPoolNLP(vocab_size, embed_size, num_classes)
mean_pool_nlp = MeanPoolNLP(vocab_size, embed_size, num_classes)

epochs = 1000
optimizer_max = optim.Adam(max_pool_nlp.parameters(), lr= 0.001)
optimizer_mean = optim.Adam(mean_pool_nlp.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

print("Training Max Pool Model...")
best_max_nlp = train_test(max_pool_nlp, loss_func, optimizer_max, train_loader, val_loader, epochs)

evaluate_model(best_max_nlp, loss_func, test_loader)

print("Training Mean Pool Model...")
best_mean_nlp = train_test(mean_pool_nlp, loss_func, optimizer_mean, train_loader, val_loader, epochs)

evaluate_model(best_mean_nlp, loss_func, test_loader)


# 4 Layer MLP Classifier in PyTorch (NLP Embedding)
Kaiming/Xavier initialization
Understanding depth challenges

* Add batch normalization or layer normalization
* dropout and regularization (L1/L2)
Gradient monitoring, learning rate scheduling, loss curves

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from copy import deepcopy

# --- 1. Robust Device Selection ---
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Check for Apple Silicon (M1/M2/M3)
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Running on: {device}")

## Define the MLPClassifer 
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, hidden_dim3, num_classes):
        # Renamed embed_layer to input_dim for clarity
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.BatchNorm1d(hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim2, hidden_dim3),
            nn.BatchNorm1d(hidden_dim3),
            nn.ReLU(),
            nn.Linear(hidden_dim3, num_classes)
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, embeddings):
        return self.layers(embeddings)
                
# define datasets      
torch.manual_seed(204)
data_size = 10000
embed_dim = 384 
hidden_dim1 = embed_dim * 4
hidden_dim2 = hidden_dim1 * 2
hidden_dim3 = hidden_dim2 
num_classes = 3 

X = torch.randn((data_size, embed_dim))
Y = torch.randint(0, num_classes, (data_size,))
X_train, X_val, X_test = torch.split(X, [8000, 1000, 1000])
Y_train, Y_val, Y_test = torch.split(Y, [8000, 1000, 1000])

train_data = TensorDataset(X_train, Y_train)
val_data = TensorDataset(X_val, Y_val)
test_data_set = TensorDataset(X_test, Y_test) # Renamed to avoid overwriting

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)
test_loader = DataLoader(test_data_set, batch_size=64) # FIX: Assigned to test_loader

# train and validate log in tensorboard
writer = SummaryWriter(log_dir="runs/mlp_stability")
model = MLPClassifier(embed_dim, hidden_dim1, hidden_dim2, hidden_dim3, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) 
l1_lambda = 1e-5
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5) 
patience = 10 
epochs = 20 
counter = 0 
best_state = None 
lowest_loss = float("inf") 
global_step = 0 

# Note: These magic commands only work in Jupyter Notebooks/Colab
# %load_ext tensorboard
# %tensorboard --logdir runs

for i in range(epochs):
    model.train()
    train_loss_acc = 0 

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        predicted = model(x)
        
        # L1 Regularization
        l1_norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(predicted, y) + l1_lambda * l1_norm 
        loss.backward()

        # Gradient Norm Monitoring
        total_grad_norm = 0.0 
        for p in model.parameters():
            if p.grad is not None: # FIX: Added safety check
                param_norm = p.grad.data.norm(2) 
                total_grad_norm += param_norm.item() ** 2 
        total_grad_norm = total_grad_norm ** 0.5 

        optimizer.step() 

        writer.add_scalar("train/loss", loss.item(), global_step)
        writer.add_scalar("train/grad_norm", total_grad_norm, global_step)

        train_loss_acc += loss.item()
        global_step += 1

    avg_train_loss = train_loss_acc / len(train_loader) 

    model.eval()
    with torch.no_grad():
        val_loss_acc = 0 

        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            predicted = model(x)
            loss = criterion(predicted, y)
            val_loss_acc += loss.item()

        avg_val_loss = val_loss_acc / len(val_loader)

    writer.add_scalar("val/avg_loss", avg_val_loss, i) 
    writer.add_scalar("train/avg_loss", avg_train_loss, i) 

    # ---- Learning rate ----
    current_lr = optimizer.param_groups[0]["lr"]
    writer.add_scalar("lr", current_lr, i)

    print(
        f"Epoch {i+1}/{epochs} | " # FIX: Changed num_epochs to epochs
        f"avg Train Loss: {avg_train_loss:.4f} | "
        f"avg val Loss: {avg_val_loss:.4f} | "
        f"LR: {current_lr:.5f}"
    )

    # FIX: Changed &lt; to <
    if avg_val_loss < lowest_loss:
        lowest_loss = avg_val_loss
        best_state = deepcopy(model.state_dict())
        counter = 0 
    else:
        counter += 1

    if counter == patience:
        print(f"early stopping at epoch {i+1}")
        break 

    # learning rate update
    scheduler.step()

writer.close()

if best_state is not None:
    model.load_state_dict(best_state) 
    
# evaluate 
model.eval()
with torch.no_grad():
    test_loss_acc = 0 
    for x, y in test_loader: # FIX: Now test_loader is defined
        x = x.to(device)
        y = y.to(device)
        predicted = model(x)
        test_loss_acc += criterion(predicted, y) 
    avg_test_loss = test_loss_acc / len(test_loader)

print(f"test avg loss is {avg_test_loss:.4f}") 
%load_ext tensorboard
%tensorboard --logdir runs

## CNN (1D conv over tokens)

convolution layers for text 
pooling strategies (max, avg)
multi filter sizes
Gradient monitoring, learning rate scheduling, loss curves
overfitting prevention, training stability 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from copy import deepcopy

# --- 1. Define Class ---
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, num_filters, kernels, dropout, num_classes):
        super().__init__()
        # Embedding: (Batch, Seq_Len) -> (Batch, Seq_Len, Embed_Dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        
        # ModuleList allows us to register a list of layers properly
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in kernels
        ])
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
        # The input to FC is num_filters * number of different kernel sizes
        self.fc = nn.Linear(num_filters * len(kernels), num_classes)

    def forward(self, input_ids):
        # 1. Embed
        embed = self.embedding(input_ids) # (B, T, D)
        
        # 2. Transpose for Conv1d: (B, T, D) -> (B, D, T)
        # Conv1d expects (Batch, Channels, Length)
        embed = embed.transpose(1, 2) 
        
        conv_outputs = []
        for conv in self.convs:
            # Apply Convolution: (B, D, T) -> (B, F, T_out)
            conv_output = conv(embed)
            
            # Apply Activation
            conv_output = self.relu(conv_output)
            
            # Global Max Pooling over time dimension: (B, F, T_out) -> (B, F)
            conv_output = torch.max(conv_output, dim=2).values 
            conv_outputs.append(conv_output)
        
        # 3. Concatenate all filter outputs: (B, F * len(kernels))
        outputs = torch.cat(conv_outputs, dim=1) 
        
        # 4. Dropout and FC
        outputs = self.dropout(outputs)
        return self.fc(outputs) 

# --- 2. Device Setup ---
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Running on: {device}")

# --- 3. Data Generation ---
data_size = 10000
batch_size = 64
seq_len = 20
vocab_size = 10000
padding_idx = 0 

# Generate random data
X = torch.randint(1, vocab_size, (data_size, seq_len))

# Add random padding (simulating variable length sequences)
padding_lens = torch.randint(0, seq_len//2, (data_size,))
for i, pad_len in enumerate(padding_lens):
    pad = pad_len.item()
    if pad > 0:
        X[i, (seq_len - pad):] = padding_idx 

# Create labels: Class 1 if any token ID >= 500, else Class 0
Y = (X >= 500).any(dim=1).long()

# Shuffle and Split
# Standard Split: 80% Train, 10% Validation, 10% Test
perm = torch.randperm(data_size)
X = X[perm]
Y = Y[perm]

X_train, X_val, X_test = torch.split(X, [8000, 1000, 1000])
Y_train, Y_val, Y_test = torch.split(Y, [8000, 1000, 1000])

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, Y_val), batch_size=batch_size) # Used for early stopping
test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size=batch_size) # Used for final check

# --- 4. Model Initialization ---
embed_dim = 128
kernel_sizes = (3, 4, 5) # Using multiple kernel sizes is common in TextCNN
dropout = 0.5
num_filters = 50

model = TextCNN(
    vocab_size, 
    embed_dim, 
    padding_idx=padding_idx, 
    num_filters=num_filters, 
    kernels=kernel_sizes, 
    dropout=dropout, 
    num_classes=2
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5) 
writer = SummaryWriter(log_dir="runs/textcnn_experiment")

# --- 5. Helper Functions ---
def get_total_accuracy(logits, labels):
    predicted = torch.argmax(logits, dim=1)
    return torch.sum((predicted == labels).float()).item()

def compute_grad_norm(model):
    total_norm = 0 
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item()**2 
    return total_norm**0.5

# --- 6. Training Loop ---
epochs = 20
patience = 5
best_state = None
lowest_loss = float("inf")
counter = 0 
global_step = 0

for epoch in range(epochs):
    # -- TRAIN --
    model.train()
    total_train_acc = 0
    
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        pred = model(x) 
        loss = criterion(pred, y)
        loss.backward()
        
        grad_norm = compute_grad_norm(model)
        optimizer.step()
        
        total_train_acc += get_total_accuracy(pred, y) 
        
        writer.add_scalar("train loss", loss.item(), global_step)
        writer.add_scalar("grad norm", grad_norm, global_step)
        global_step += 1
        
    avg_train_accuracy = total_train_acc / len(X_train)

    # -- VALIDATION (For Early Stopping) --
    model.eval()
    total_val_loss = 0
    total_val_acc = 0 
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            pred = model(x) 
            loss = criterion(pred, y) 
            
            # FIX: Use loss.item() to avoid memory leak
            total_val_loss += loss.item()
            total_val_acc += get_total_accuracy(pred, y) 

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_accuracy = total_val_acc / len(X_val)

    # Logging
    writer.add_scalar("avg_train_acc", avg_train_accuracy, epoch)
    writer.add_scalar("avg_val_acc", avg_val_accuracy, epoch) 
    writer.add_scalar("avg_val_loss", avg_val_loss, epoch) 
    writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

    print(f"Epoch {epoch+1} | Train Acc: {avg_train_accuracy:.4f} | Val Acc: {avg_val_accuracy:.4f} | Val Loss: {avg_val_loss:.4f}")

    # -- Early Stopping Logic --
    # FIX: Replaced &lt; with <
    if avg_val_loss < lowest_loss:
        lowest_loss = avg_val_loss 
        best_state = deepcopy(model.state_dict())
        counter = 0
        print("  -> New best model found.")
    else:
        counter += 1
        print(f"  -> No improvement. Patience {counter}/{patience}")

    if counter == patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

    scheduler.step(avg_val_loss)

writer.close()

# --- 7. Final Evaluation on Test Set ---
if best_state is not None:
    model.load_state_dict(best_state)    

model.eval()
test_loss = 0
test_acc = 0 

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        
        test_loss += loss.item() 
        test_acc += get_total_accuracy(pred, y) 

print("-" * 30)
print(f"Final Test Loss: {test_loss/len(test_loader):.4f}")
print(f"Final Test Acc:  {test_acc/len(X_test):.4f}")
print("-" * 30)

%load_ext tensorboard
%tensorboard --logdir runs --port 6007

Running on: mps
Epoch 1 | Train Acc: 0.9894 | Val Acc: 1.0000 | Val Loss: 0.0009
  -> New best model found.
Epoch 2 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0006
  -> New best model found.
Epoch 3 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0005
  -> New best model found.
Epoch 4 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0004
  -> New best model found.
Epoch 5 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0004
  -> New best model found.
Epoch 6 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0004
  -> New best model found.
Epoch 7 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0004
  -> New best model found.
Epoch 8 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0003
  -> New best model found.
Epoch 9 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0003
  -> New best model found.
Epoch 10 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.0003
  -> New best model found.
Epoch 11 | Train Acc: 1.0000 | Val Acc: 1.0000 | Val Loss: 0.00

# RNN/GRU/LSTM


vanilla RNN 
GRU
LSTM 
Bidirectional variants
Gradient monitoring, learning rate scheduling, loss curves
overfitting prevention, training stability 

In [52]:
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_size, padding_idx, memory_cell_class, hidden_dim, num_layers, bidirectional, representation, num_classes):
        super().__init__()
        self.padding_idx = padding_idx
        self.bidirectional = bidirectional
        self.representation = representation
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=padding_idx)
        
        # Instantiate the passed class (nn.GRU or nn.LSTM)
        self.memory_cell = memory_cell_class(
            embed_size,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True
        )
        
        multiplier = 2 if bidirectional else 1
        self.FC = nn.Linear(multiplier * hidden_dim, num_classes)

    def forward(self, input_ids, lengths):
        # input_ids: (B, T)
        embedding = self.embedding(input_ids) 
        
        # Pack
        packed_embedding = pack_padded_sequence(
            embedding,
            lengths=lengths.cpu(), # Must be on CPU for packing
            batch_first=True,
            enforce_sorted=False
        )
        
        # Forward pass through RNN
        if isinstance(self.memory_cell, nn.LSTM):
            packed_outputs, (hidden_layers, cell_layers) = self.memory_cell(packed_embedding)
        else:
            packed_outputs, hidden_layers = self.memory_cell(packed_embedding)
            
        # hidden_layers shape: (num_layers * num_directions, B, hidden_dim)

        if self.representation == "final":
            # Handle bidirectional stacking
            if self.bidirectional:
                # -2 is the last forward layer, -1 is the last backward layer
                forward_hidden = hidden_layers[-2]
                backward_hidden = hidden_layers[-1]
                final_representation = torch.cat([forward_hidden, backward_hidden], dim=1)
            else:
                final_representation = hidden_layers[-1]
                
        elif self.representation == "mean":
            # Unpack
            # CRITICAL FIX: total_length ensures output matches input_ids shape even if batch max len is shorter
            padded_outputs, _ = pad_packed_sequence(
                packed_outputs, 
                batch_first=True, 
                total_length=input_ids.size(1) 
            ) 
            
            # Masking
            mask = (input_ids != self.padding_idx).unsqueeze(-1).float() 
            masked_sum = (mask * padded_outputs).sum(dim=1)
            
            # Division (lengths is on device because it came from x)
            # Clamp lengths to avoid division by zero
            final_representation = masked_sum / lengths.unsqueeze(1).float().clamp(min=1.0)
            
        return self.FC(final_representation)

# --- Setup ---

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

# Synthetic Data 
vocab_size = 1000
dataset_size = 10000
seq_len = 20
padding_idx = 0 
embed_dim = 256
num_classes = 2 
batch_size = 64

X = torch.randint(1, vocab_size, (dataset_size, seq_len))
all_paddings = torch.randint(0, seq_len//2, (dataset_size,))
for i, padding in enumerate(all_paddings):
    pad = padding.item()
    if pad > 0:
        X[i, (seq_len - pad):] = padding_idx 
Y = (X >= 450).any(dim=1).long()

X_train, X_val, X_test = torch.split(X, [8000,1000,1000])
Y_train, Y_val, Y_test = torch.split(Y, [8000,1000,1000])

train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)
test_dataset = TensorDataset(X_test, Y_test)

# Optimization: Use num_workers > 0
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=2) 

# Hyperparameters
hidden_dim = 128
num_layers = 1
bidirectional = True # Changed to True to test that logic
representation = "mean" # Changed to mean to test that logic
epochs = 10 
patience = 3

# model = RNNClassifier(
#     vocab_size, embed_dim, padding_idx, nn.LSTM, # Passing LSTM class
#     hidden_dim, num_layers, bidirectional, representation, num_classes
# ).to(device)
# model = RNNClassifier(
#     vocab_size, embed_dim, padding_idx, nn.GRU, # Passing LSTM class
#     hidden_dim, num_layers, bidirectional, representation, num_classes
# ).to(device)
model = RNNClassifier(
    vocab_size, embed_dim, padding_idx, nn.RNN, # Passing LSTM class
    hidden_dim, num_layers, bidirectional, representation, num_classes
).to(device)



optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5) 
criterion = nn.CrossEntropyLoss()

def get_total_accuracy(logits, label):
    preds = torch.argmax(logits, dim=1)
    return (preds == label).float().sum().item()

# Training Loop
writer = SummaryWriter(log_dir="runs/model_runs")
global_step = 0
best_state = None
min_loss = float("inf")
counter = 0

for epoch in range(epochs):
    model.train()
    avg_train_acc = 0
    
    for x, y in train_loader:
        optimizer.zero_grad()
        x, y = x.to(device), y.to(device)
        
        # Calculate lengths and clamp to avoid 0
        lengths = (x != padding_idx).sum(dim=1).clamp(min=1)
        
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        
        # Optimization: clip_grad_norm_ returns the norm, no need to recalc
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        
        optimizer.step()
        
        avg_train_acc += get_total_accuracy(logits, y) 
        writer.add_scalar("train/grad_norm", grad_norm, global_step) 
        writer.add_scalar("train/loss", loss.item(), global_step) 
        global_step += 1

    avg_train_acc /= len(train_dataset)
    writer.add_scalar("train/avg_acc", avg_train_acc, epoch+1) 

    # Validation
    avg_val_acc = 0
    avg_val_loss = 0 
    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            lengths = (x != padding_idx).sum(dim=1).clamp(min=1)
            
            logits = model(x, lengths)
            loss = criterion(logits, y) 
            
            avg_val_acc += get_total_accuracy(logits, y)
            avg_val_loss += loss.item()

    avg_val_acc /= len(val_dataset)
    avg_val_loss /= len(val_loader)
    
    writer.add_scalar("val/avg_acc", avg_val_acc, epoch+1)
    writer.add_scalar("val/avg_loss", avg_val_loss, epoch+1)
    writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch+1) 

    print(f"Epoch {epoch+1}: Train Acc {avg_train_acc:.4f} | Val Acc {avg_val_acc:.4f} | Val Loss {avg_val_loss:.4f}")
    
    scheduler.step(avg_val_loss)

    if avg_val_loss < min_loss:
        min_loss = avg_val_loss
        best_state = deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

# Testing
model.eval()
avg_test_acc = 0
avg_test_loss = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        lengths = (x != padding_idx).sum(dim=1).clamp(min=1)
        
        logits = model(x, lengths)
        loss = criterion(logits, y)
        
        avg_test_loss += loss.item()
        avg_test_acc += get_total_accuracy(logits, y)

avg_test_acc /= len(test_dataset)
avg_test_loss /= len(test_loader)

print(f"Final Test Loss: {avg_test_loss:.4f} | Accuracy: {avg_test_acc:.4f}")

%load_ext tensorboard
%tensorboard --logdir runs --port 6007


Using device: mps
Epoch 1: Train Acc 0.9981 | Val Acc 1.0000 | Val Loss 0.0000
Epoch 2: Train Acc 0.9999 | Val Acc 1.0000 | Val Loss 0.0001
Epoch 3: Train Acc 0.9998 | Val Acc 1.0000 | Val Loss 0.0002
Epoch 4: Train Acc 0.9999 | Val Acc 1.0000 | Val Loss 0.0000
Early stopping at epoch 4
Final Test Loss: 0.0000 | Accuracy: 1.0000
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## Attention 

bahdanau/luong attention 
attention scoring functions 
context vectors

Gradient monitoring, learning rate scheduling, loss curves
overfitting prevention, training stability 
handling variable length sequences 

Attention visualization, positional information

## Transformer (single-head)
self-attention mechanism 
positional encoding 
feed forward networks
Attention visualization, positional information

## Transformer ( multi-head)

multi attention heads 
full encoder/decoder
modern architectures (BERT-style, GPT-style)
Attention visualization, positional information